# 02 — Analyse request body

Load `request_body` from `mitm_http_captures` and explore structure step by step.

Same component path as notebook 01. Run from repo root.

In [1]:
import json
from collections import Counter
from urllib.parse import urlparse

from pprint import pprint
from sqlalchemy import text

from core.db import SessionLocal

component_path = "/flagship-web/rsc-action/actions/component"

query = text("""
    SELECT request_url, request_body
    FROM mitm_http_captures
    WHERE request_body IS NOT NULL
    ORDER BY captured_at_ms DESC
""")

with SessionLocal() as session:
    rows = session.execute(query).fetchall()

component_rows = [
    (url, body)
    for url, body in rows
    if urlparse(url).path == component_path
]

print(len(rows), "rows with a request body")
print(len(component_rows), "rows on component path")
print(component_rows[0][1][:500])

518 rows with a request body
326 rows on component path
{"clientArguments":{"payload":{"jobId":"4417153139","isTwoPane":false,"hideInterestCard":false},"states":[],"requestMetadata":{"$type":"proto.sdui.common.RequestMetadata"},"screenId":"com.linkedin.sdui.flagshipnav.jobs.JobDetails"}}


## top-level keys

In [2]:
bodies = [json.loads(body) for _, body in component_rows]

for body in bodies[:3]:
    print(body.keys())
    print("-" * 40)

print(Counter(frozenset(body.keys()) for body in bodies))

dict_keys(['clientArguments'])
----------------------------------------
dict_keys(['clientArguments'])
----------------------------------------
dict_keys(['clientArguments'])
----------------------------------------
Counter({frozenset({'clientArguments'}): 326})


## clientArguments

In [3]:
client_args = [body["clientArguments"] for body in bodies]

for args in client_args[:3]:
    print(args.keys())
    print("-" * 40)

print(Counter(frozenset(args.keys()) for args in client_args))

dict_keys(['payload', 'states', 'requestMetadata', 'screenId'])
----------------------------------------
dict_keys(['payload', 'states', 'requestMetadata', 'screenId'])
----------------------------------------
dict_keys(['payload', 'states', 'requestMetadata', 'screenId'])
----------------------------------------
Counter({frozenset({'states', 'payload', 'requestMetadata', 'screenId'}): 326})


## payload keys

In [4]:
payload_key_counter = Counter()

for args in client_args:
    payload = args.get("payload")
    if isinstance(payload, dict):
        payload_key_counter.update(payload.keys())

pprint(payload_key_counter.most_common())

[('jobId', 326),
 ('isTwoPane', 135),
 ('renderAsCard', 69),
 ('hideInterestCard', 33),
 ('isPremium', 33),
 ('profilePicture', 3),
 ('profileUrl', 3),
 ('isTopApplicant', 3),
 ('companyLogo', 3),
 ('tooltipLegoTrackingToken', 3)]


## jobId

In [5]:
job_ids = [args["payload"]["jobId"] for args in client_args if "jobId" in args.get("payload", {})]

print(len(job_ids), "rows with jobId")
print(len(set(job_ids)), "distinct jobIds")
print(job_ids[:10])

326 rows with jobId
25 distinct jobIds
['4417153139', '4417153139', '4417153139', '4417153139', '4417153139', '4417153139', '4417153139', '4417153139', '4417153139', '4417153139']


## states

In [6]:
states_lengths = Counter(len(args.get("states", [])) for args in client_args)

print(states_lengths)
print()
print("sample states values:")
for args in client_args[:5]:
    print(args.get("states"))

Counter({0: 326})

sample states values:
[]
[]
[]
[]
[]


## requestMetadata

In [7]:
metadata_counter = Counter(
    json.dumps(args.get("requestMetadata"), sort_keys=True) for args in client_args
)

print(len(metadata_counter), "distinct requestMetadata shapes")
pprint(metadata_counter.most_common(5))

1 distinct requestMetadata shapes
[('{"$type": "proto.sdui.common.RequestMetadata"}', 326)]


## screenId

In [8]:
pprint(Counter(args.get("screenId") for args in client_args).most_common())

[('com.linkedin.sdui.flagshipnav.jobs.JobDetails', 310),
 ('com.linkedin.sdui.flagshipnav.jobs.SemanticJobDetails', 16)]
